# 020. LSTM/GRU input/output shape

- return_sequences = False, True 일 때의 output 비교

- return_state = False, True 일 때의 internal state output 비교

- Bidirectional LSTM/GRU 의 output 비교

In [1]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Bidirectional, GRU
import numpy as np

T = 4   #Time Steps
D = 2   #features
U = 3   #LSTM units

X = np.random.randn(2, T, D)
print(X.shape)
X

(2, 4, 2)


array([[[-0.22178673,  0.91856338],
        [-0.08067287, -0.77376868],
        [-0.34538095, -0.57556657],
        [-1.13430517, -0.74161928]],

       [[ 0.73536726, -1.96945964],
        [-0.89886678,  0.85382088],
        [ 0.06333219, -0.5595934 ],
        [ 1.55344997,  0.07502444]]])

# LSTM

## return_sequences

- False (default) - last time step 의 output 만 반환
- True - 모든 timestep 의 output 을 모두 반환

In [2]:
def lstm(return_sequences=False):
    input_ = Input(shape=(T, D))
    rnn = LSTM(U, return_sequences=return_sequences)
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)

    return model.predict(X)

print("---- return_sequences=False ----> last timestep 의 outout only")
lstm_out = lstm(return_sequences=False)
print(lstm_out.shape)
print(lstm_out)
print("\n---- return_sequences=True ----> 모든 timestep 별 output")
lstm_out = lstm(return_sequences=True)
print(lstm_out.shape)
print(lstm_out)

---- return_sequences=False ----> last timestep 의 outout only
(2, 3)
[[ 0.00435411  0.01482638  0.07390508]
 [-0.22941706  0.0535823  -0.12077547]]

---- return_sequences=True ----> 모든 timestep 별 output
(2, 4, 3)
[[[ 0.01040476  0.07202391  0.15791368]
  [ 0.02430126  0.03264387  0.01876162]
  [ 0.04320074  0.05044589 -0.02036672]
  [ 0.09205668  0.09948829 -0.02256937]]

 [[-0.03285662 -0.03810264 -0.10087211]
  [ 0.03388684  0.11177031  0.02946992]
  [ 0.03393576  0.04149649 -0.03968022]
  [-0.12307168 -0.20279592 -0.1789759 ]]]


## return_state

- False (default) - output 만 반환

- True - output, last step 의 hidden state, cell state (LSTM 의 경우) 반환

In [3]:
def lstm(return_state=False):
    input_ = Input(shape=(T, D))
    rnn = LSTM(U, return_state=return_state)
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    
    if return_state:
        o, h, c = model.predict(X)
        print("o :", o, o.shape)
        print("h :", h, h.shape)
        print("c :", c, c.shape)
    else:
        o = model.predict(X)
        print("o :", o, o.shape)

print("---- return_state=False ----> outout only")       
lstm(return_state=False)
print("\n---- return_state=True ----> outout, hidden state, cell state all")      
lstm(return_state=True)

---- return_state=False ----> outout only
o : [[ 0.07744133 -0.13702884 -0.13940924]
 [-0.15887722  0.04854202 -0.05164766]] (2, 3)

---- return_state=True ----> outout, hidden state, cell state all
o : [[-0.41511622 -0.17116576 -0.02354242]
 [ 0.05476552  0.09505524  0.25955752]] (2, 3)
h : [[-0.41511622 -0.17116576 -0.02354242]
 [ 0.05476552  0.09505524  0.25955752]] (2, 3)
c : [[-0.6786336  -0.68993485 -0.05863827]
 [ 0.16902801  0.15170696  0.46194413]] (2, 3)


# Bidirectional LSTM

- 순방향, 역방향이 concatenate 된 output 출력  

- hidden state, cell state 는 순방향, 역방향 별도 출력

In [8]:
T, D, U

(4, 2, 3)

In [10]:
def lstm(return_sequences=False, return_state=False):
    input_ = Input(shape=(T, D))
    rnn = Bidirectional(LSTM(U, return_state=return_state, return_sequences=return_sequences))
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    
    if return_state:    
        o, h1, c1, h2, c2 = model.predict(X)
        print("o :", o, o.shape)
        print("h1 :", h1, h1.shape)
        print("c1 :", c1, c1.shape)
        print("h2 :", h2, h2.shape)
        print("c2 :", c2, c2.shape)
    else:
        o = model.predict(X)
        print("o :", o, o.shape)

print("---- return_sequences = False ----> 순방향, 역방향이 concatenate 된 output")
lstm()
print()
print("---- return_sequences = True ----")
lstm(return_sequences=True)
print()
print("---- return_state = True ----")
lstm(return_state=True)

---- return_sequences = False ----> 순방향, 역방향이 concatenate 된 output
o : [[ 0.05909878 -0.09889865  0.00657569  0.00821178  0.05971782 -0.0010341 ]
 [ 0.2642725  -0.16177478  0.08030604 -0.05481743  0.2548585   0.01253955]] (2, 6)

---- return_sequences = True ----
o : [[[-0.05200402 -0.02138821 -0.1020173  -0.04482364  0.03452854
    0.00968574]
  [-0.0138245   0.00351752 -0.03010238 -0.21868595  0.19593298
    0.10994225]
  [-0.0120515   0.00644563 -0.01339981 -0.15386298  0.12126771
    0.03646242]
  [-0.02607053 -0.00743176 -0.01675852 -0.08047479  0.07918739
   -0.00559932]]

 [[ 0.0382648   0.11246717  0.04354212 -0.06188926  0.19379301
    0.21296813]
  [-0.04405866  0.01280926 -0.0719948   0.06363802 -0.03093557
    0.0094158 ]
  [-0.01365782  0.0283894  -0.02188581 -0.02932903  0.08151932
    0.13483682]
  [ 0.08847658  0.09236152  0.06021363  0.05679033  0.0117146
    0.06600215]]] (2, 4, 6)

---- return_state = True ----
o : [[ 0.19427961  0.25019357 -0.12288144  0.03504758 -0

# GRU 

- cell state 가 없는 것만 LSTM 과 차이

In [13]:
def gru(return_sequences=False, return_state=False):
    input_ = Input(shape=(T, D))
    rnn = GRU(U, return_state=return_state, return_sequences=return_sequences)
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    
    if return_state:    
        o, h = model.predict(X)
        print("o :", o, o.shape)
        print("h :", h, h.shape)
    else:
        o = model.predict(X)
        print("o :", o, o.shape)

print("---- return_sequences = False ----")
gru()
print()
print("---- return_sequences = True ----")
gru(return_sequences=True)
print()
print("---- return_state = True ----")
gru(return_state=True)

---- return_sequences = False ----
o : [[ 0.16262816  0.36622688 -0.14584513]
 [ 0.22956511 -0.09585768 -0.00557957]] (2, 3)

---- return_sequences = True ----
o : [[[ 0.17144887  0.21394864 -0.3576045 ]
  [-0.13416837 -0.04636942  0.03585007]
  [-0.10199781 -0.20471941  0.22204737]
  [ 0.12703806 -0.47531298  0.4316292 ]]

 [[-0.741184   -0.12737148  0.16215132]
  [-0.3117397   0.08361642 -0.19251738]
  [-0.31219804 -0.03118216  0.04206149]
  [-0.5547115   0.09921995 -0.11642619]]] (2, 4, 3)

---- return_state = True ----
o : [[-0.09821232  0.29236874 -0.05668249]
 [ 0.31060958 -0.48218164  0.12528417]] (2, 3)
h : [[-0.09821232  0.29236874 -0.05668249]
 [ 0.31060958 -0.48218164  0.12528417]] (2, 3)


# Bidirectional GRU

- cell state 가 없는 것 외에 LSTM 과 동일

In [14]:
def gru(return_sequences=False, return_state=False):
    input_ = Input(shape=(T, D))
    rnn = Bidirectional(GRU(U, return_state=return_state, return_sequences=return_sequences))
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    if return_state:    
        o, h1, h2 = model.predict(X)
        print("o :", o, o.shape)
        print("h1 :", h1, h1.shape)
        print("h2 :", h2, h2.shape)
    else:
        o = model.predict(X)
        print("o :", o, o.shape)

print("---- return_sequences = False ----")
gru(return_sequences=False)
print()
print("---- return_sequences = True ----")
gru(return_sequences=True)
print()
print("---- return_state = True ----")
gru(return_state=True)

---- return_sequences = False ----
o : [[ 0.5154484  -0.27070478  0.43450227 -0.0380442  -0.07714257  0.18243009]
 [-0.10177675 -0.1014176  -0.09691557 -0.24532786  0.11860459 -0.38038138]] (2, 6)

---- return_sequences = True ----
o : [[[-0.23483425 -0.1314209   0.12779461  0.00759377 -0.03911881
    0.03477535]
  [ 0.126903    0.10075425 -0.12847543  0.31772324 -0.43787393
   -0.01984501]
  [ 0.09189203  0.15125544 -0.2510577   0.34543455 -0.39052895
    0.13077442]
  [-0.20883138  0.21377581 -0.42053002  0.4415139  -0.36953557
    0.24173667]]

 [[ 0.58559626  0.3474176  -0.17027628  0.3316478  -0.40674853
   -0.4939487 ]
  [-0.17338483  0.19259852 -0.07663418 -0.04623397  0.03199746
   -0.01428524]
  [ 0.08790638  0.18762136 -0.13002762  0.07022134 -0.1577762
   -0.26747182]
  [ 0.2825576  -0.16725175  0.5062053  -0.12279131  0.25404936
   -0.42011952]]] (2, 4, 6)

---- return_state = True ----
o : [[ 0.4944959   0.3966694   0.33340454 -0.07274694 -0.1063491  -0.04018281]
 [-0.3691